# DeepSORVF — Colab Setup

This notebook sets up the DeepSORVF pipeline on Google Colab.

**Steps:**
1. Mount Google Drive
2. Install dependencies
3. Verify weights + clips
4. Run a test inference

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/DeepSORVF_Project'
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# Step 2: Install dependencies
!pip install ultralytics==8.4.121
!pip install filterpy lap easydict geopy pyproj fastdtw loguru
!pip install scikit-image

import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

In [ ]:
# Step 3: Verify weights
import os

weights_dir = os.path.join(PROJECT_ROOT, 'weights')
required = {
    'best.pt': 40_000_000,        # ~42 MB (YOLOv8)
    'YOLOX-final.pth': 30_000_000, # ~34 MB (YOLOX)
    'ckpt.t7': 40_000_000,         # ~44 MB (DeepSORT ReID)
}

all_ok = True
for name, min_size in required.items():
    path = os.path.join(weights_dir, name)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1_000_000
        status = '✓' if size_mb > min_size / 1_000_000 else '✗ too small'
        print(f'  {name}: {size_mb:.1f} MB {status}')
    else:
        print(f'  {name}: MISSING ✗')
        all_ok = False

print(f'\nWeights: {"OK" if all_ok else "MISSING FILES"}')

In [ ]:
# Step 4: Verify clips on Google Drive
clips = ['clip-01', 'clip-02', 'clip-10', 'Video-10', 'Video-28', 'Video-29', 'Video-34']

for clip in clips:
    clip_path = os.path.join(PROJECT_ROOT, clip)
    if os.path.isdir(clip_path):
        files = os.listdir(clip_path)
        video = [f for f in files if f.endswith('.mp4') or f.endswith('.avi')]
        ais_dir = os.path.join(clip_path, 'ais')
        ais_count = len(os.listdir(ais_dir)) if os.path.isdir(ais_dir) else 0
        print(f'  {clip}: video={video[0] if video else "MISSING"}, ais_files={ais_count}')
    else:
        print(f'  {clip}: NOT FOUND ✗')

print('\nDone.')

In [ ]:
# Step 5: Quick test — load YOLOX + YOLOv8
import sys
sys.path.insert(0, PROJECT_ROOT)

from detection_yolox.yolo import YOLO
yolo = YOLO()
print('YOLOX loaded ✓')

from detection_yolov8.yolov8_detector import YOLOv8Detector, ULTRALYTICS_OK
if ULTRALYTICS_OK:
    yolov8 = YOLOv8Detector(weights=os.path.join(PROJECT_ROOT, 'weights/best.pt'), conf=0.30)
    print('YOLOv8 loaded ✓')
else:
    print('YOLOv8 not available')

In [ ]:
# Step 6: Run ablation on clip-01 (200 frames, C5 = full pipeline)
import os
os.chdir(PROJECT_ROOT)

from run_ablation import run_pipeline

stats = run_pipeline(
    clip_name='clip-01',
    result_dir='/tmp/colab_test/',
    config_name='C5_colab_test',
    max_frames=200
)
print(f'\nResult: {stats}')

## Ready!

The pipeline is set up. You can now:
- Run ablation configs: `python run_ablation.py --clip clip-01 --config C5 --max-frames 200`
- Batch run all configs: `python ablation_runner.py`
- Run interactive notebooks in `notebooks/`